# Unidad 4 · Tablas de contingencia y chi-cuadrado

Cuaderno de la base abierta de ingeniería (Análisis de Datos). Reproduce el ejemplo resuelto (cafetería) y la pieza del caso Estampa Sur. Lee los CSV de `../datos/`. Las cifras se comparan con `verificacion/u04_chi_cuadrado.py`.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import stats
from statsmodels.stats.contingency_tables import Table

DATOS = Path("../datos")
RES = {}

def v_cramer(chi2, tabla):
    return np.sqrt(chi2 / (tabla.sum() * (min(tabla.shape) - 1)))

## 1. Ejemplo · ¿el tipo de pedido depende de la franja horaria?

In [ ]:
ej = pd.read_csv(DATOS / "u04_ejemplo.csv").set_index("franja")
print(ej)
print((ej.div(ej.sum(axis=1), axis=0) * 100).round(1))       # porcentaje de cada fila
chi = stats.chi2_contingency(ej.values)                        # en tablas mayores que 2 x 2 no hay Yates
print("esperados:\n", chi.expected_freq.round(2))
print("chi2 =", round(chi.statistic, 2), " gl =", chi.dof, " p =", round(chi.pvalue, 4))
print("V de Cramér =", round(v_cramer(chi.statistic, ej.values), 3))
print("residuos ajustados:\n", Table(ej.values).standardized_resids.round(2))
RES.update(ej_chi2=chi.statistic, ej_p=chi.pvalue, ej_V=v_cramer(chi.statistic, ej.values))

## 2. Caso Estampa Sur · ¿el tipo de reclamo depende del canal?

In [ ]:
rc = pd.read_csv(DATOS / "reclamos_canal.csv").set_index("canal")
print(rc)
print((rc.div(rc.sum(axis=1), axis=0) * 100).round(1))
chi = stats.chi2_contingency(rc.values)
print("esperado mínimo:", chi.expected_freq.min().round(2))
print("chi2 =", round(chi.statistic, 2), " gl =", chi.dof, " p =", f"{chi.pvalue:.2g}")
V = v_cramer(chi.statistic, rc.values)
print("V de Cramér =", round(V, 3))
aj = Table(rc.values).standardized_resids
print(pd.DataFrame(aj, index=rc.index, columns=rc.columns).round(2))
RES.update(caso_chi2=chi.statistic, caso_p=chi.pvalue, caso_gl=chi.dof, caso_V=V, caso_ajustados=np.asarray(aj))

In [ ]:
import matplotlib.pyplot as plt
plt.imshow(aj, cmap="coolwarm", vmin=-4, vmax=4)
plt.xticks(range(3), rc.columns); plt.yticks(range(3), rc.index)
plt.colorbar(label="residuo ajustado"); plt.title("Dónde está la asociación"); plt.show()

## 3. Prueba exacta de Fisher (tablas de 2 x 2 con esperados chicos)

In [ ]:
print(stats.fisher_exact([[1, 9], [5, 3]]))   # ejercicio 4.10